# Notebook 2 — Results & Analysis

This notebook reproduces and analyses all key results:
1. Training loss curves (baseline CNN vs fine-tuned ResNet-18)
2. Classification report — EuroSAT (block split)
3. Confusion matrix — EuroSAT & UC Merced
4. Change detection — ROC curve & threshold justification
5. Spatial leakage experiment summary
6. Error analysis — top-5 misclassified tiles with hypotheses
7. Embedding visualisation (t-SNE) — Bonus C

In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

RUNS = Path('../runs')

def load_json(path):
    p = Path(path)
    return json.loads(p.read_text()) if p.exists() else None

print('Runs directory:', RUNS.resolve())

## 1. Training Loss Curves

In [ ]:
# Baseline CNN
baseline_history = load_json(RUNS / 'baseline/history.json') or []

# Fine-tuned ResNet-18 (two phases)
resnet_history = load_json(RUNS / 'resnet18/history.json') or {}
phase1 = resnet_history.get('phase1', [])
phase2 = resnet_history.get('phase2', [])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training Loss Curves', fontsize=14, fontweight='bold')

# Baseline
if baseline_history:
    ep  = [d['epoch'] for d in baseline_history]
    trl = [d['train_loss'] for d in baseline_history]
    vll = [d['val_loss']   for d in baseline_history]
    axes[0].plot(ep, trl, 'o-', label='Train')
    axes[0].plot(ep, vll, 's--', label='Val')
    axes[0].set_title('Baseline Scratch CNN')
    axes[0].legend(); axes[0].grid(alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'No baseline history found\nRun train_baseline.py first',
                 ha='center', va='center', transform=axes[0].transAxes)

for ax, phase, title in [
    (axes[1], phase1, 'ResNet-18 Phase 1 (frozen backbone)'),
    (axes[2], phase2, 'ResNet-18 Phase 2 (last 2 blocks unfrozen)'),
]:
    if phase:
        ep  = [d['epoch'] for d in phase]
        trl = [d['train_loss'] for d in phase]
        vll = [d['val_loss']   for d in phase]
        ax.plot(ep, trl, 'o-', label='Train')
        ax.plot(ep, vll, 's--', label='Val')
        ax.legend(); ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, 'No history found\nRun train_finetune.py first',
                ha='center', va='center', transform=ax.transAxes)
    ax.set_title(title)

for ax in axes:
    ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy loss')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(RUNS / 'resnet18/loss_curves_combined.png', dpi=150)
plt.show()

## 2. Classification Report — EuroSAT

In [ ]:
report_csv = RUNS / 'resnet18/eurosat_eval/classification_report.csv'
if report_csv.exists():
    df = pd.read_csv(report_csv, index_col=0)
    display_cols = ['precision', 'recall', 'f1-score', 'support']
    df_show = df[[c for c in display_cols if c in df.columns]]
    print('EuroSAT Classification Report (block split):')
    print(df_show.to_string(float_format='{:.4f}'.format))

    # Bar chart of per-class F1
    class_rows = df_show.drop(index=[i for i in df_show.index if 'avg' in str(i) or i == 'accuracy'], errors='ignore')
    fig, ax = plt.subplots(figsize=(11, 4))
    class_rows['f1-score'].plot.bar(ax=ax, color='steelblue')
    ax.set_title('Per-class F1 — EuroSAT (ResNet-18, block split)', fontsize=12, fontweight='bold')
    ax.set_ylabel('F1 Score')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=35)
    ax.axhline(class_rows['f1-score'].mean(), color='red', linestyle='--', label='Macro mean')
    ax.legend()
    plt.tight_layout()
    plt.savefig(RUNS / 'resnet18/eurosat_eval/per_class_f1.png', dpi=150)
    plt.show()
else:
    print('Run evaluate.py --data-dir data/eurosat first.')

## 3. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, img_path, title in [
    (axes[0], RUNS / 'resnet18/eurosat_eval/confusion_matrix.png', 'EuroSAT (block split)'),
    (axes[1], RUNS / 'resnet18/uc_merced_eval/confusion_matrix.png', 'UC Merced holdout (21 classes)'),
]:
    p = Path(img_path)
    if p.exists():
        ax.imshow(plt.imread(str(p)))
        ax.set_title(title, fontsize=12, fontweight='bold')
    else:
        ax.text(0.5, 0.5, f'Not found:\n{p.name}\nRun evaluate.py first',
                ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Change Detection — ROC Curve & Threshold

In [ ]:
roc_path = RUNS / 'change_detection/roc_curve.png'
cd_summary = load_json(RUNS / 'change_detection/summary.json')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

if roc_path.exists():
    axes[0].imshow(plt.imread(str(roc_path)))
    axes[0].set_title('ROC Curve — Embedding Change Detector', fontsize=11, fontweight='bold')
    axes[0].axis('off')
else:
    axes[0].text(0.5, 0.5, 'Run run_change_detection.py first', ha='center', va='center',
                 transform=axes[0].transAxes)
    axes[0].axis('off')

if cd_summary:
    axes[1].axis('off')
    info = (
        f"Change Detection Summary\n"
        f"{'─'*35}\n"
        f"Pair count         : {cd_summary.get('pair_count', '?'):,}\n"
        f"Mean similarity    : {cd_summary.get('mean_similarity', 0):.4f}\n"
        f"Selected threshold : {cd_summary.get('selected_similarity_threshold', 0):.4f}\n"
        f"  (Youden-J = argmax(TPR - FPR))\n\n"
        f"Threshold interpretation:\n"
        f"  sim < threshold  -> CHANGED\n"
        f"  sim >= threshold -> UNCHANGED"
    )
    axes[1].text(0.05, 0.95, info, ha='left', va='top', transform=axes[1].transAxes,
                 fontsize=11, family='monospace',
                 bbox=dict(boxstyle='round', facecolor='#EBF5FB', alpha=0.8))
else:
    axes[1].text(0.5, 0.5, 'No change detection summary found', ha='center', va='center',
                 transform=axes[1].transAxes)
    axes[1].axis('off')

plt.tight_layout()
plt.show()

## 5. Change Heatmaps — 5 Region Pairs

In [ ]:
heatmap_dir = RUNS / 'change_detection/heatmaps'
heatmap_files = sorted(heatmap_dir.glob('heatmap_pair_*.png')) if heatmap_dir.exists() else []

if heatmap_files:
    for hf in heatmap_files[:5]:
        fig, ax = plt.subplots(figsize=(13, 4))
        ax.imshow(plt.imread(str(hf)))
        ax.set_title(hf.stem, fontsize=10)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
else:
    print('Run scripts/save_change_heatmaps.py first.')

## 6. Spatial Leakage Experiment

In [ ]:
leakage_img = RUNS / 'spatial_leakage/split_comparison_f1.png'
block_report  = load_json(RUNS / 'spatial_leakage/block_report.json')
random_report = load_json(RUNS / 'spatial_leakage/random_report.json')

if leakage_img.exists():
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.imshow(plt.imread(str(leakage_img)))
    ax.axis('off')
    plt.tight_layout()
    plt.show()

if block_report and random_report:
    block_macro  = block_report['macro avg']['f1-score']
    random_macro = random_report['macro avg']['f1-score']
    print(f'Block-split  macro-F1 : {block_macro:.4f}')
    print(f'Random-split macro-F1 : {random_macro:.4f}')
    print(f'Delta (random - block): {random_macro - block_macro:+.4f}')
    if random_macro - block_macro > 0.01:
        print('=> Spatial leakage inflates random-split accuracy.')
    else:
        print('=> Minimal spatial leakage effect detected.')
else:
    print('Run scripts/spatial_leakage_experiment.py first.')

## 7. Error Analysis — Top-5 Misclassifications

In [ ]:
error_grid = RUNS / 'resnet18/error_analysis/top_misclassifications_grid.png'
errors_json = load_json(RUNS / 'resnet18/error_analysis/top_misclassifications.json')

if error_grid.exists():
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.imshow(plt.imread(str(error_grid)))
    ax.axis('off')
    plt.tight_layout()
    plt.show()

if errors_json:
    df_err = pd.DataFrame(errors_json)
    print('Top-5 high-confidence misclassifications:')
    print(df_err[['true', 'predicted', 'confidence']].to_string(float_format='{:.4f}'.format, index=False))
else:
    print('Run scripts/visualize_errors.py first.')

## 8. Embedding Visualisation — t-SNE (Bonus C)

In [ ]:
# NOTE: This cell extracts embeddings from the full EuroSAT dataset and projects
# them to 2D with t-SNE.  Requires a trained checkpoint.
# Expected runtime: ~5 min on CPU for 27k samples.

CHECKPOINT = RUNS / 'resnet18/best.pt'
EUROSAT_DIR = Path('../data/eurosat')

if not CHECKPOINT.exists():
    print('Checkpoint not found — run train_finetune.py first.')
else:
    from landuse.data import load_image_folder, make_loader
    from landuse.models import ResNetEmbeddingExtractor, load_checkpoint
    from landuse.change import extract_embeddings
    from sklearn.manifold import TSNE

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = load_image_folder(EUROSAT_DIR, train=False)
    checkpoint = torch.load(CHECKPOINT, map_location=device)
    classes = checkpoint.get('classes', dataset.classes)
    model = load_checkpoint(str(CHECKPOINT), num_classes=len(classes), device=device)
    extractor = ResNetEmbeddingExtractor(model).to(device)
    loader = make_loader(dataset, batch_size=64)

    print('Extracting embeddings ...')
    embeddings, labels = extract_embeddings(extractor, loader, device)
    print(f'Embeddings shape: {embeddings.shape}')

    print('Running t-SNE (this may take a few minutes) ...')
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    proj = tsne.fit_transform(embeddings)

    fig, ax = plt.subplots(figsize=(12, 9))
    palette = plt.cm.get_cmap('tab10', len(classes))
    for i, cls in enumerate(classes):
        mask = labels == i
        ax.scatter(proj[mask, 0], proj[mask, 1], c=[palette(i)],
                   label=cls, alpha=0.4, s=8, linewidths=0)
    ax.legend(markerscale=3, fontsize=8, loc='best')
    ax.set_title('t-SNE of ResNet-18 Embeddings — EuroSAT (27k tiles)', fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    out_path = RUNS / 'resnet18/tsne_embeddings.png'
    plt.savefig(out_path, dpi=150)
    plt.show()
    print(f'Saved to {out_path}')